# Fine-tuning de Gemma-7b-it con QLoRA/SFT — versión notebook

Mismo pipeline QLoRA/SFT del laboratorio Lecture04b, con `google/gemma-7b-it`
en vez de Qwen3-8B. Gemma ya está soportado de fábrica por la imagen base de
Hugging Face (transformers 4.42), así que este pipeline evita la cadena de
incompatibilidades de versiones que tocó resolver para Qwen3.

**Adaptado para correr sobre el contenedor JupyterHub de la clase 04**
(`quay.io/jupyter/tensorflow-notebook:x86_64-cuda-latest`, ver `docker-compose.yml`
de este lab). Esta es la versión notebook, celda por celda, de `train_gemma.py`
— mismo pipeline, mismos valores por defecto; úsala para explorar/depurar
interactivamente, y `train_gemma.py` para correr el entrenamiento completo de
una sola vez (por ejemplo desde una terminal de Jupyter).

## Qué cambia frente a la versión pensada para Vertex AI

1. **Faltan dependencias.** Esta imagen es la línea "tensorflow-notebook" de
   Jupyter Docker Stacks: trae TensorFlow + CUDA, pero **no** trae PyTorch ni
   el stack de Hugging Face (a diferencia de la línea "pytorch-notebook", que
   sí lo incluiría). La celda de instalación (abajo) se encarga de esto — el
   usuario `jovyan` tiene permisos de escritura sobre `/opt/conda`, así que no
   hace falta `sudo` ni `--user`.
2. **La salida ya no va a `AIP_MODEL_DIR`** (esa variable es de Vertex AI
   Training y no existe aquí). Los adaptadores se guardan por defecto en
   `/home/jovyan/labs/...` — la carpeta que el `docker-compose.yml` de este lab
   monta desde `$HOME/si7016-262` del host, así que sobreviven a que borres o
   reinicies el contenedor.
3. **Login de Hugging Face explícito e interactivo** (celda de abajo, con
   `getpass` — nunca hardcodees el token en el notebook).

## Requisito importante: Gemma es un modelo "gated"

1. Entra a https://huggingface.co/google/gemma-7b-it y acepta la licencia con tu cuenta de HF.
2. Genera un token de acceso en https://huggingface.co/settings/tokens.
3. Pégalo cuando la celda de login te lo pida (sección 1).

Sin esto, la descarga del modelo falla con un error 401/403 "gated repo".


## 0. Instalar dependencias que la imagen base no trae

In [1]:
%pip install torch transformers peft trl bitsandbytes accelerate datasets huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 60.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 40.3 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 60.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 56.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 54.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 51.1 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 85.4 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 83.3 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2

## 1. Login en Hugging Face (interactivo, sin hardcodear el token)

In [2]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
login(token=hf_token)
del hf_token  # no lo dejamos flotando en una variable más de lo necesario


Token de Hugging Face (con acceso a google/gemma-7b-it):  ········


## 2. Configuración

Valores por defecto idénticos a `train_gemma.py`. Ajusta lo que necesites
directamente aquí (en el `.py` sería vía `argparse`).


In [ ]:
import os
import types

args = types.SimpleNamespace(
    model_name="google/gemma-7b-it",
    data_path="./corpus/qa.jsonl",              # Archivo JSONL
    eval_split_size=0.1,                  # 10% para evaluación
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=-1,                        # -1 para que complete las épocas sobre el split de train
    output_dir=os.environ.get("OUTPUT_DIR", "/home/jovyan/labs/gemma-crobotp-lora"),
)

print(f"Modelo base: {args.model_name}")
print(f"Archivo de datos: {args.data_path}")
print(f"Directorio de salida: {args.output_dir}")


Modelo base: google/gemma-7b-it
Archivo de datos: datos.jsonl
Directorio de salida: /home/jovyan/labs/gemma-crobotp-lora


## 3. Verificar GPU

In [4]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: no se detectó GPU. Revisa que el contenedor se haya levantado "
          "con --gpus all / runtime: nvidia (ver docker-compose.yml de este lab) "
          "y que `nvidia-smi` funcione dentro del contenedor.")


CUDA disponible: True
GPU: NVIDIA L4


## 4. Cargar el modelo base en 4-bit

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    args.model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(args.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Memoria GPU ocupada tras cargar el modelo (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "N/A")


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Memoria GPU ocupada tras cargar el modelo (GB): 5.58


## 5. Adaptadores LoRA

Mismos `target_modules` que en el notebook original — Gemma usa la misma
convención de nombres de proyecciones que Llama/Qwen.


In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=args.lora_r,
    lora_alpha=args.lora_alpha,
    lora_dropout=args.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 50,003,968 || all params: 8,587,684,864 || trainable%: 0.5823


## 6. Dataset + formateo con chat template

Nota: a diferencia de Qwen3, la plantilla de chat de Gemma no tiene modo
"thinking", así que NO pasamos `enable_thinking` aquí (evita cualquier duda
sobre kwargs específicos de un solo modelo).


In [8]:
from datasets import load_dataset

# 1. Cargar el dataset completo desde tu archivo datos.jsonl
full_dataset = load_dataset("json", data_files=args.data_path, split="train")

# 2. Realizar el Split en Train y Evaluation (con semilla fija para reproducibilidad)
dataset_split = full_dataset.train_test_split(
    test_size=args.eval_split_size, 
    seed=42
)

raw_train_dataset = dataset_split["train"]
raw_eval_dataset = dataset_split["test"]

print(f"Ejemplos para entrenamiento: {len(raw_train_dataset)}")
print(f"Ejemplos para evaluación: {len(raw_eval_dataset)}")

# 3. Función de formateo con la plantilla oficial de Gemma
def format_crobotp_data(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

# 4. Aplicar el formateo a ambos conjuntos
train_dataset = raw_train_dataset.map(format_crobotp_data, remove_columns=raw_train_dataset.column_names)
eval_dataset = raw_eval_dataset.map(format_crobotp_data, remove_columns=raw_eval_dataset.column_names)

print("\n--- Muestra del primer ejemplo de Train ---")
print(train_dataset[0]["text"][:400])


Generating train split: 0 examples [00:00, ? examples/s]

Ejemplos para entrenamiento: 693
Ejemplos para evaluación: 77


Map:   0%|          | 0/693 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]


--- Muestra del primer ejemplo de Train ---
<bos><start_of_turn>user
¿Qué parámetro define el tiempo de gas previo al encendido del arco (Pre-gas)?<end_of_turn>
<start_of_turn>model
El parámetro de tiempo de Pre-gas en la configuración de la función de soldadura.<end_of_turn>



## 7. Entrenamiento con `SFTTrainer`

Los checkpoints intermedios se quedan en el disco efímero del contenedor
(`/tmp/...`) — solo los adaptadores finales (sección 8) se guardan en
`args.output_dir`, que sí persiste en el host vía el volumen montado.


In [10]:
from trl import SFTConfig, SFTTrainer

local_ckpt_dir = "/tmp/gemma-7b-crobotp-ckpts"

sft_config = SFTConfig(
    output_dir=local_ckpt_dir,
    per_device_train_batch_size=args.per_device_train_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    learning_rate=args.learning_rate,
    logging_steps=10,
    eval_strategy="epoch",            # Evalúa el modelo al final de cada época
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,        # Se añade el conjunto de evaluación
)

# Iniciar entrenamiento con monitoreo de eval_loss
trainer.train()

Adding EOS to train dataset:   0%|          | 0/693 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/693 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/693 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/693 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/693 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/77 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/77 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/77 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.654333,2.243653,1.682614,37361.000000,0.572486
2,1.812758,1.999990,1.776460,74722.000000,0.594099
3,1.041217,1.939821,1.214632,112083.000000,0.616742


TrainOutput(global_step=132, training_loss=2.5838240619861716, metrics={'train_runtime': 681.2996, 'train_samples_per_second': 3.052, 'train_steps_per_second': 0.194, 'total_flos': 5695959958603776.0, 'train_loss': 2.5838240619861716, 'epoch': 3.0})

## 8. Guardar los adaptadores LoRA en el volumen montado

In [12]:
os.makedirs(args.output_dir, exist_ok=True)
trainer.save_model(args.output_dir)
tokenizer.save_pretrained(args.output_dir)
print(f"Adaptadores guardados en: {args.output_dir}")
print("(esa carpeta vive dentro del volumen montado -sigue disponible en "
      "$HOME/si7016-262 del host aunque borres el contenedor)")


Adaptadores guardados en: /home/jovyan/labs/gemma-crobotp-lora
(esa carpeta vive dentro del volumen montado -sigue disponible en $HOME/si7016-262 del host aunque borres el contenedor)


## Notas finales

- Para la corrida completa (no la prueba rápida de 20 pasos), vuelve a la
  sección 2 y pon `max_steps=-1` (así se respeta `num_train_epochs=3`
  completas).
- Este mismo pipeline existe como script plano en `train_gemma.py`, útil para
  lanzarlo de una sola vez desde una terminal (`python train_gemma.py --hf_token ...`)
  en vez de correrlo celda por celda.
- `requirements.txt` (en esta misma carpeta) lista las dependencias que la
  celda 0 instaló — útil si prefieres instalarlas por fuera del notebook
  (`pip install -r requirements.txt`) antes de abrirlo.
